# Whisper Realtime Computer Audio Demo

Notebook kiểm thử nhận dạng giọng nói realtime bằng Whisper, lấy nguồn âm thanh trực tiếp từ máy tính (loopback/stereo mix).

**Outline:**
1. Cài đặt và nhập các thư viện cần thiết
2. Tạo module thu âm thanh từ máy tính
3. Tạo module xử lý và truyền dữ liệu âm thanh
4. Tích hợp Whisper để nhận dạng giọng nói theo thời gian thực
5. Notebook kiểm thử: Thu và nhận dạng âm thanh realtime

## 1. Cài đặt và nhập các thư viện cần thiết
Cài đặt sounddevice, numpy, whisper, và các phụ thuộc khác.

## Chuẩn bị: Kích hoạt Stereo Mix trên Windows

Để ghi âm được audio hệ thống (YouTube, Spotify, browser, etc.), bạn cần kích hoạt **Stereo Mix**:

**Trên Windows:**
1. Open Sound settings: `Settings > System > Sound > More volume options`
2. Scroll to **Recording devices** và click "Show disabled devices"
3. Right-click **Stereo Mix** → Enable
4. Set nó làm default recording device nếu cần

**Nếu không thấy Stereo Mix:**
- Cập nhật audio driver từ trang hãng sản xuất
- Hoặc dùng Virtual Audio Cable (VB-Audio Virtual Cable) thay thế

**Trên Linux/macOS:**
- Linux: Dùng `pavucontrol` hoặc setup loopback device
- macOS: Dùng Blackhole hoặc SoundFlower virtual audio

In [ ]:
# Cài đặt các thư viện cần thiết
# %pip install -q sounddevice numpy torch openai-whisper

import sounddevice as sd
import numpy as np
import whisper
import threading
import queue
import sys
from collections import deque
from datetime import datetime

print('sounddevice:', sd.__version__)
print('numpy:', np.__version__)
print('whisper:', whisper.__version__)
print('Python:', sys.version)

sounddevice: 0.5.5
numpy: 1.26.4
whisper: 20250625
Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


## 2. Tạo module thu âm thanh từ máy tính
Sử dụng sounddevice để thu âm thanh từ thiết bị đầu ra (loopback/stereo mix).

In [6]:
# Module thu âm thanh từ thiết bị đầu ra (loopback/stereo mix)
import sounddevice as sd
import numpy as np
import platform

def find_stereo_mix_device():
    """Tìm Stereo Mix device trên Windows hoặc loopback device trên Linux/Mac"""
    devices = sd.query_devices()
    
    # Tìm stereo mix (Windows)
    stereo_mix_keywords = ['stereo mix', 'what u hear', 'loopback', 'virtual audio cable']
    
    for i, device in enumerate(devices):
        if device['max_input_channels'] > 0:  # Chỉ input devices
            device_name = device['name'].lower()
            if any(keyword in device_name for keyword in stereo_mix_keywords):
                print(f"✓ Found: Device {i} - {device['name']}")
                return i
    
    print("⚠️ Stereo Mix device not found!")
    print("On Windows, enable Stereo Mix in Sound Settings or use Virtual Audio Cable.")
    print("You can select another device below...\n")
    return None

def list_audio_devices():
    """Hiển thị danh sách thiết bị audio"""
    print("\n=== Available Recording Devices ===")
    devices = sd.query_devices()
    input_devices = []
    for i, device in enumerate(devices):
        if device['max_input_channels'] > 0:
            input_devices.append((i, device))
            print(f"[{i}] {device['name']}")
            print(f"    Channels: {device['max_input_channels']}, Sample Rate: {device['default_samplerate']} Hz\n")
    return input_devices

def select_audio_device():
    """Cho phép user chọn device thủ công"""
    input_devices = list_audio_devices()
    
    # Nếu có ít nhất 1 device
    if input_devices:
        while True:
            try:
                choice = input(f"Select device [0-{len(input_devices)-1}] (or press Enter for default): ").strip()
                if choice == "":
                    # Dùng device default
                    print("✓ Using default recording device")
                    return None
                device_id = int(choice)
                if 0 <= device_id < len(input_devices):
                    selected_device_id = input_devices[device_id][0]
                    device_name = input_devices[device_id][1]['name']
                    print(f"✓ Selected: [{selected_device_id}] {device_name}\n")
                    return selected_device_id
                else:
                    print(f"Invalid choice. Please select 0-{len(input_devices)-1}")
            except ValueError:
                print("Invalid input. Please enter a number.")
    else:
        print("❌ No recording devices found!")
        return None

# Hiển thị thiết bị có sẵn
STEREO_MIX_DEVICE = find_stereo_mix_device()

# Nếu không tìm thấy Stereo Mix, cho user chọn device
if STEREO_MIX_DEVICE is None:
    print("\n" + "="*60)
    print("Select a recording device manually:")
    print("="*60)
    SELECTED_DEVICE = select_audio_device()
else:
    SELECTED_DEVICE = STEREO_MIX_DEVICE
    print(f"\n✓ Will use Stereo Mix device: {STEREO_MIX_DEVICE}\n")

⚠️ Stereo Mix device not found!
On Windows, enable Stereo Mix in Sound Settings or use Virtual Audio Cable.
You can select another device below...


Select a recording device manually:

=== Available Recording Devices ===
[0] Microsoft Sound Mapper - Input
    Channels: 2, Sample Rate: 44100.0 Hz

[1] CABLE Output (VB-Audio Virtual 
    Channels: 16, Sample Rate: 44100.0 Hz

[5] Primary Sound Capture Driver
    Channels: 2, Sample Rate: 44100.0 Hz

[6] CABLE Output (VB-Audio Virtual Cable)
    Channels: 16, Sample Rate: 44100.0 Hz

[12] CABLE Output (VB-Audio Virtual Cable)
    Channels: 2, Sample Rate: 48000.0 Hz

[13] CABLE Output (VB-Audio Point)
    Channels: 16, Sample Rate: 44100.0 Hz

[15] Input (VB-Audio Point)
    Channels: 16, Sample Rate: 44100.0 Hz

✓ Selected: [1] CABLE Output (VB-Audio Virtual 



In [7]:
# ===== AUDIO HELPER FUNCTIONS + TEST DEVICES =====

def get_device_channels(device_id):
    """Lay so channel phu hop cho device (max 2)"""
    info = sd.query_devices(device_id)
    return min(info['max_input_channels'], 2)

def get_native_sr(device_id):
    """Lay sample rate native cua device"""
    return int(sd.query_devices(device_id)['default_samplerate'])

def audio_to_mono(audio_data):
    """Chuyen multi-channel audio thanh mono"""
    if audio_data.ndim == 2 and audio_data.shape[1] > 1:
        return audio_data.mean(axis=1).astype(np.float32)
    return audio_data.flatten().astype(np.float32)

def resample_audio(audio, orig_sr, target_sr=16000):
    """Resample audio ve 16000 Hz cho Whisper (numpy interpolation)"""
    if orig_sr == target_sr:
        return audio
    n_samples = int(len(audio) * target_sr / orig_sr)
    resampled = np.interp(
        np.linspace(0, len(audio), n_samples),
        np.arange(len(audio)),
        audio
    )
    return resampled.astype(np.float32)

def test_device_audio(device_id, duration=2):
    """Test device: ghi am tai native SR, resample ve 16kHz, tinh RMS"""
    try:
        device_info = sd.query_devices(device_id)
        num_ch = get_device_channels(device_id)
        native_sr = get_native_sr(device_id)
        print(f"\nTesting device [{device_id}]: {device_info['name']}")
        print(f"   Channels: {num_ch} | Native SR: {native_sr} Hz")
        print(f"   Recording {duration}s... ", end="", flush=True)

        audio = sd.rec(
            int(duration * native_sr),
            samplerate=native_sr,
            channels=num_ch,
            dtype='float32',
            device=device_id
        )
        sd.wait()

        audio = audio_to_mono(audio)
        if native_sr != 16000:
            audio = resample_audio(audio, native_sr, 16000)

        rms = np.sqrt(np.mean(audio**2))
        peak = np.max(np.abs(audio))
        print("Done!")
        print(f"   RMS: {rms:.4f} | Peak: {peak:.4f}", end="  ")

        if rms < 0.01:
            print("Very quiet (noise floor)")
            return False
        elif rms < 0.05:
            print("Quiet (increase volume?)")
            return True
        else:
            print("Good audio level!")
            return True

    except Exception as e:
        print(f"   Error: {e}")
        return False

# === Test tat ca devices ===
print("\n" + "="*60)
print("TEST AUDIO DEVICES (native SR + resample to 16kHz)")
print("="*60)
devices = sd.query_devices()
test_results = {}

for i, device in enumerate(devices):
    if device['max_input_channels'] > 0:
        has_audio = test_device_audio(i, duration=2)
        test_results[i] = (device['name'], has_audio)

print("\n" + "="*60)
print("SUMMARY:")
print("="*60)
for dev_id, (dev_name, has_audio) in test_results.items():
    status = "HAS AUDIO" if has_audio else "NO AUDIO "
    print(f"[{dev_id}] {dev_name:<42} {status}")
print("="*60 + "\n")


TEST AUDIO DEVICES (native SR + resample to 16kHz)

Testing device [0]: Microsoft Sound Mapper - Input
   Channels: 2 | Native SR: 44100 Hz
   Recording 2s... Done!
   RMS: 0.0801 | Peak: 0.3972  Good audio level!

Testing device [1]: CABLE Output (VB-Audio Virtual 
   Channels: 2 | Native SR: 44100 Hz
   Recording 2s... Done!
   RMS: 0.0594 | Peak: 0.4171  Good audio level!

Testing device [5]: Primary Sound Capture Driver
   Channels: 2 | Native SR: 44100 Hz
   Recording 2s... Done!
   RMS: 0.0552 | Peak: 0.3987  Good audio level!

Testing device [6]: CABLE Output (VB-Audio Virtual Cable)
   Channels: 2 | Native SR: 44100 Hz
   Recording 2s... Done!
   RMS: 0.0595 | Peak: 0.3774  Good audio level!

Testing device [12]: CABLE Output (VB-Audio Virtual Cable)
   Channels: 2 | Native SR: 48000 Hz
   Recording 2s... Done!
   RMS: 0.0443 | Peak: 0.3159  Quiet (increase volume?)

Testing device [13]: CABLE Output (VB-Audio Point)
   Channels: 2 | Native SR: 44100 Hz
   Recording 2s... Done

## 3. Xử lý và truyền dữ liệu âm thanh
Chuyển đổi dữ liệu sang định dạng phù hợp cho Whisper, chia đoạn nếu cần.

In [8]:
# Hàm chuẩn hóa audio
def preprocess_audio(audio, target_sr=16000):
    """Chuẩn hóa và điều chỉnh biên độ audio"""
    if audio.dtype != np.float32:
        audio = audio.astype(np.float32)
    
    # Chuẩn hóa biên độ tránh bị cắt
    max_val = np.max(np.abs(audio))
    if max_val > 1e-5:  # Tránh chia cho 0
        audio = audio / max_val * 0.95
    return audio

# Hàm chia đoạn (chunk) cho streaming realtime
def chunk_audio(audio, chunk_size=16000*5):  # 5s mỗi chunk
    """Chia audio thành các chunk nhỏ để streaming"""
    chunks = []
    for i in range(0, len(audio), chunk_size):
        chunk = audio[i:i+chunk_size]
        if len(chunk) > 0:
            chunks.append(chunk)
    return chunks

# Class quản lí audio buffer cho streaming realtime
class AudioRingBuffer:
    """Ring buffer để lưu trữ audio stream realtime"""
    def __init__(self, samplerate=16000, buffer_duration=10):
        self.samplerate = samplerate
        self.buffer_size = int(buffer_duration * samplerate)
        self.buffer = deque(maxlen=self.buffer_size)
    
    def add_audio(self, audio_chunk):
        """Thêm audio chunk vào buffer"""
        if isinstance(audio_chunk, np.ndarray):
            audio_chunk = audio_chunk.flatten()
        for sample in audio_chunk:
            self.buffer.append(sample)
    
    def get_buffer(self):
        """Lấy toàn bộ buffer hiện tại"""
        return np.array(list(self.buffer), dtype=np.float32)
    
    def clear(self):
        """Xóa buffer"""
        self.buffer.clear()

## 4. Tích hợp Whisper để nhận dạng giọng nói theo thời gian thực
Sử dụng mô hình Whisper để nhận dạng từng đoạn audio.

In [9]:
# Hàm nhận dạng realtime với Whisper
import whisper
import warnings
warnings.filterwarnings('ignore')

class RealtimeTranscriber:
    """Transcriber realtime cho audio streaming"""
    def __init__(self, model_name="small", language="vi", device="cpu"):
        self.model = whisper.load_model(model_name, device=device)
        self.language = language
        self.device = device
        self.transcribe_queue = queue.Queue()
        self.results = []
        self.lock = threading.Lock()
        
    def transcribe_audio_streaming(self, audio_buffer, chunk_duration=5):
        """Transcribe audio với streaming chunks"""
        samplerate = 16000
        chunk_size = int(chunk_duration * samplerate)
        
        if len(audio_buffer) < chunk_size:
            return  # Chưa đủ dữ liệu
        
        # Lấy chunk từ đầu buffer
        audio_chunk = audio_buffer[:chunk_size]
        audio_chunk = preprocess_audio(audio_chunk)
        
        # Transcribe
        try:
            result = self.model.transcribe(
                audio_chunk, 
                language=self.language, 
                fp16=False, 
                task="transcribe", 
                verbose=False
            )
            text = result["text"].strip()
            if text:  # Chỉ lưu nếu có text
                with self.lock:
                    self.results.append(text)
                return text
        except Exception as e:
            print(f"Transcription error: {e}")
            return None
    
    def get_full_transcript(self):
        """Lấy toàn bộ transcript đã transcribe"""
        with self.lock:
            return " ".join(self.results)

def transcribe_chunks(chunks, model_name="small", language="vi", device="cpu"):
    """Function cũ - transcribe batch chunks từ file audio đã ghi"""
    model = whisper.load_model(model_name, device=device)
    results = []
    for i, chunk in enumerate(chunks):
        print(f"Transcribing chunk {i+1}/{len(chunks)}...")
        result = model.transcribe(chunk, language=language, fp16=False, task="transcribe", verbose=False)
        text = result["text"].strip()
        if text:
            print(f"  ✓ {text}")
            results.append(text)
    return results

## 5. Notebook kiểm thử: Thu và nhận dạng âm thanh realtime
Ghi âm từ máy tính, chia đoạn, nhận dạng và hiển thị kết quả.

## Cách hoạt động Realtime Streaming:

1. **Audio Capture**: Thu âm thanh từ Stereo Mix với sounddevice callback
2. **Ring Buffer**: Lưu trữ stream audio vào circular buffer (giảm memory)
3. **Streaming Chunks**: Mỗi 3 giây, lấy chunk từ buffer để transcribe
4. **Threading**: Transcription chạy trên thread riêng, không block recording
5. **Live Display**: Hiển thị text khi vừa transcribe (latency ~5-10 giây)
6. **Final Transcript**: Kết hợp tất cả chunks vào transcript cuối cùng

**Tối ưu:**
- Reduce `CHUNK_DURATION` để latency thấp hơn (nhưng CPU cao hơn)
- Dùng `"tiny"` model nếu CPU yếu (14ms inference vs 40ms cho "small")
- Tăng `buffer_duration` nếu có dropout audio

In [22]:
# ============ REALTIME STREAMING DEMO (SỬA LỖI BUFFER SLIDING WINDOW) ============
import time

# === CAU HINH ===
DURATION = 30          # Thoi gian ghi am (giay)
SAMPLERATE = 16000     # Whisper yeu cau 16kHz (buffer se o 16kHz sau resample)
MODEL_NAME = "small"  # medium/large cho chất lượng tốt hơn
LANGUAGE = "vi"
CHUNK_DURATION = 4     # Chunk dài hơn để transcript tự nhiên hơn
BLOCK_SIZE = 4096
CHUNK_OVERLAP = 1      # Overlap 1s giữa các chunk

# Tu dong detect thong so device
_num_channels = get_device_channels(SELECTED_DEVICE)
_native_sr = get_native_sr(SELECTED_DEVICE)

print(f"\n{'='*60}")
print("REALTIME AUDIO TRANSCRIPTION - Whisper (Sliding buffer fix)")
print(f"{'='*60}")
print(f"Model: {MODEL_NAME} | Language: {LANGUAGE} | Duration: {DURATION}s")
print(f"Device: [{SELECTED_DEVICE}] {sd.query_devices(SELECTED_DEVICE)['name']}")
print(f"Channels: {_num_channels} | Native SR: {_native_sr} Hz -> 16000 Hz")
print(f"Transcribe every: {CHUNK_DURATION}s, overlap: {CHUNK_OVERLAP}s")
print(f"{'='*60}\n")
print("Loading Whisper model (first time may take ~1 min)...")

transcriber = RealtimeTranscriber(model_name=MODEL_NAME, language=LANGUAGE, device="cpu")
audio_buffer = AudioRingBuffer(samplerate=SAMPLERATE)
print("Model loaded!\n")

# Biến nhớ tổng số sample đã thu (không reset khi buffer đầy)
total_samples = 0
last_transcribed_sample = 0

def audio_callback(indata, frames, time_info, status):
    global total_samples
    if status:
        print(f"Status: {status}")
    # Convert to mono
    if indata.ndim == 2 and indata.shape[1] > 1:
        audio_data = indata.mean(axis=1).astype(np.float32)
    else:
        audio_data = indata.flatten().astype(np.float32)
    # Resample to 16000 Hz neu can
    if _native_sr != 16000:
        audio_data = resample_audio(audio_data, _native_sr, 16000)
    audio_buffer.add_audio(audio_data)
    total_samples += len(audio_data)

def has_sentence_end(text):
    """Kiểm tra text có kết thúc bằng dấu câu không"""
    return any(text.strip().endswith(punct) for punct in ['.', '?', '!', '。', '！', '？'])

def streaming_transcriber():
    global last_transcribed_sample
    buffer_len_needed = int(SAMPLERATE * CHUNK_DURATION)
    overlap_len = int(SAMPLERATE * CHUNK_OVERLAP)
    last_text = ""
    buffer_size = audio_buffer.buffer.maxlen if hasattr(audio_buffer.buffer, 'maxlen') else len(audio_buffer.get_buffer())
    while not stop_event.is_set():
        buffer_data = audio_buffer.get_buffer()
        cur_total = total_samples
        # Tính vị trí start/end tương đối với buffer hiện tại
        buffer_start_sample = cur_total - len(buffer_data)
        start = last_transcribed_sample - buffer_start_sample
        end = start + buffer_len_needed
        # Nếu start < 0 thì sample đã bị loại khỏi buffer, bỏ qua
        if start >= 0 and end <= len(buffer_data):
            audio_chunk = buffer_data[start:end]
            timestamp = datetime.now().strftime("%H:%M:%S")
            text = transcriber.transcribe_audio_streaming(audio_chunk, chunk_duration=CHUNK_DURATION)
            if text:
                if has_sentence_end(text) or (text.strip() != last_text.strip()):
                    print(f"[{timestamp}] {text}")
                    last_text = text
            else:
                print(f"[{timestamp}] (no speech)")
            # Cập nhật vị trí đã transcribe, giữ lại overlap
            last_transcribed_sample += buffer_len_needed - overlap_len
        time.sleep(0.1)

stop_event = threading.Event()
transcriber_thread = threading.Thread(target=streaming_transcriber, daemon=True)
transcriber_thread.start()

print(f"Recording started... ({DURATION}s, play audio now!)\n")

try:
    with sd.InputStream(
        device=SELECTED_DEVICE,
        samplerate=_native_sr,
        channels=_num_channels,
        blocksize=BLOCK_SIZE,
        callback=audio_callback
    ):
        sd.sleep(int(DURATION * 1000))
except KeyboardInterrupt:
    print("\nRecording interrupted")
except Exception as e:
    print(f"\nError: {e}")
finally:
    stop_event.set()
    transcriber_thread.join(timeout=3)

print(f"\n{'='*60}")
print("FINAL TRANSCRIPT:")
print(f"{'='*60}")
full_text = transcriber.get_full_transcript()
print(full_text if full_text else "(No speech detected)")
print(f"{'='*60}")


REALTIME AUDIO TRANSCRIPTION - Whisper (Sliding buffer fix)
Model: small | Language: vi | Duration: 30s
Device: [1] CABLE Output (VB-Audio Virtual 
Channels: 2 | Native SR: 44100 Hz -> 16000 Hz
Transcribe every: 4s, overlap: 1s

Loading Whisper model (first time may take ~1 min)...
Model loaded!

Recording started... (30s, play audio now!)



100%|██████████| 400/400 [00:02<00:00, 142.28frames/s]


[11:17:07] Chưa sử dụng quyết định này và vẫn đang thực hiện biệt kết hối các khu nghiệp bình thường trên toàn.


100%|██████████| 400/400 [00:02<00:00, 146.27frames/s]


[11:17:10] đối với các người bình thường trên toàn quốc. Chỉ lưu ý rằng chúng ta có giá trị của riêng mươi.


...

[11:17:25] Bán điều hành của chúng ta là tôi hiện tại...


100%|██████████| 400/400 [00:01<00:00, 246.75frames/s]


[11:17:28] Tôi hiện tại vẫn hỗ trợ các chị là leader team.


100%|██████████| 400/400 [00:01<00:00, 222.73frames/s]

[11:17:31] Để tìm chết sở Mên chết sở

FINAL TRANSCRIPT:
Chưa sử dụng quyết định này và vẫn đang thực hiện biệt kết hối các khu nghiệp bình thường trên toàn. đối với các người bình thường trên toàn quốc. Chỉ lưu ý rằng chúng ta có giá trị của riêng mươi. là có giá trị của riêng mình, mình sẽ sử dụng cái quyết định. Sử dụng quyết định này vào những việc sâu xa hơn để các chị sẽ tình dung ra. thì sẽ dùng ra cách làm chúng ta rất là dễ dàng và mang lại nhịp. và mang lại nhiều kết quả Đây là cái mang điều hành của chúng ta Bán điều hành của chúng ta là tôi hiện tại... Tôi hiện tại vẫn hỗ trợ các chị là leader team. Để tìm chết sở Mên chết sở


In [ ]:
# ============ DEBUG MODE ============
import time

_debug_channels = get_device_channels(SELECTED_DEVICE)
_debug_native_sr = get_native_sr(SELECTED_DEVICE)

print(f"\n{'='*60}")
print("DEBUG MODE - Audio Level & Whisper Detection")
print(f"{'='*60}")
print(f"Device: [{SELECTED_DEVICE}] {sd.query_devices(SELECTED_DEVICE)['name']}")
print(f"Channels: {_debug_channels} | Native SR: {_debug_native_sr} Hz")
print(f"Model: medium | Recording 10s, checking every 2s...")
print(f"{'='*60}\n")
print("Loading model...")

transcriber_debug = RealtimeTranscriber(model_name="medium", language="vi", device="cpu")
audio_buffer_debug = AudioRingBuffer(samplerate=16000, buffer_duration=10)
print("Model loaded!\n")

def audio_callback_debug(indata, frames, time_info, status):
    if status:
        print(f"Status: {status}")
    if indata.ndim == 2 and indata.shape[1] > 1:
        audio_data = indata.mean(axis=1).astype(np.float32)
    else:
        audio_data = indata.flatten().astype(np.float32)
    if _debug_native_sr != 16000:
        audio_data = resample_audio(audio_data, _debug_native_sr, 16000)
    audio_buffer_debug.add_audio(audio_data)

stop_event_debug = threading.Event()
print(f"Recording... (play audio now!)\n")

try:
    with sd.InputStream(
        device=SELECTED_DEVICE,
        samplerate=_debug_native_sr,
        channels=_debug_channels,
        blocksize=BLOCK_SIZE,
        callback=audio_callback_debug
    ):
        for i in range(5):
            time.sleep(2)
            buffer_data = audio_buffer_debug.get_buffer()
            buffer_size = len(buffer_data)

            if buffer_size > 0:
                rms = np.sqrt(np.mean(buffer_data**2))
                peak = np.max(np.abs(buffer_data))
                print(f"[{i+1}] {buffer_size:6d} samples | RMS: {rms:.4f} | Peak: {peak:.4f}", end="")

                if buffer_size >= 16000 * 2:
                    chunk = preprocess_audio(buffer_data[-16000*2:])
                    try:
                        result = transcriber_debug.model.transcribe(
                            chunk, language="vi", fp16=False, task="transcribe", verbose=False
                        )
                        text = result["text"].strip()
                        print(f" | {'Detected: ' + text[:45] if text else 'No speech'}")
                    except Exception as e:
                        print(f" | Error: {str(e)[:30]}")
                else:
                    print()
            else:
                print(f"[{i+1}] No data in buffer!")

except KeyboardInterrupt:
    print("\nStopped")
except Exception as e:
    print(f"\nError: {e}")

print(f"\n{'='*60}")
print("RMS < 0.01 -> Device khong bat duoc audio")
print("RMS 0.01-0.05 -> Audio nho, tang volume len")
print("RMS > 0.05 -> Audio OK!")
print("Detected text -> Whisper hoat dong tot!")
print(f"{'='*60}\n")


🐛 DEBUG MODE - Audio Level & Whisper Detection
Device: 7
Recording 10s, checking audio every 2s...

🔴 Recording... (playing YouTube audio now!)

[1] No data in buffer!
[2] No data in buffer!
[3] No data in buffer!
[4] No data in buffer!
[5] No data in buffer!

Debug Summary:
If RMS < 0.01: Device không bắt audio từ YouTube
If RMS > 0.05: Audio OK nhưng Whisper không detect (hoặc text bị empty)
If Whisper detected text: Audio & Whisper OK



## Advanced: Các tuỳ chọn tối ưu

**Giảm latency:**
```python
CHUNK_DURATION = 2          # Từ 3s xuống 2s (faster response, higher CPU)
MODEL_NAME = "tiny"         # Nhanh hơn nhưng accuracy thấp hơn
SAMPLERATE = 16000         # Không đổi, Whisper yêu cầu 16kHz
```

**Tăng accuracy:**
```python
MODEL_NAME = "medium"       # Accuracy cao hơn nhưng slow hơn
LANGUAGE = "vi"             # Specify language để tăng accuracy
```

**Xử lý nhiều ngôn ngữ:**
```python
# Để model auto-detect language:
result = self.model.transcribe(audio_chunk, fp16=False, verbose=False)
# language parameter sẽ được detect tự động
```

**Lưu transcript ra file:**
```python
full_text = transcriber.get_full_transcript()
with open("transcript.txt", "w", encoding="utf-8") as f:
    f.write(full_text)
print("✓ Transcript saved to transcript.txt")
```

In [30]:
# ===== TEST WHISPER WITH SAMPLE AUDIO =====
# Nếu device không capture được, test Whisper bằng audio sample
# Để xác nhận model có hoạt động không

print("\n" + "="*60)
print("🧪 TEST: Whisper Recognition with Sample Audio")
print("="*60)

# Tạo sample audio: 2 giây tiếng nói (440Hz sine wave - simulate speech pattern)
duration = 3
sample_rate = 16000
t = np.linspace(0, duration, int(sample_rate * duration), False)

# Tạo audio pattern giả hành động như tiếng nói
# Kết hợp nhiều tần số để simulate như human voice
signal = (
    0.2 * np.sin(2 * np.pi * 200 * t) +  # Base frequency
    0.1 * np.sin(2 * np.pi * 300 * t) +  # Mid frequency
    0.05 * np.sin(2 * np.pi * 500 * t)   # High frequency
)

# Normalize
signal = signal / np.max(np.abs(signal)) * 0.9
signal = signal.astype(np.float32)

print(f"\nGenerated test audio: {duration}s at {sample_rate}Hz")
print(f"Audio shape: {signal.shape}")
print(f"Audio level: RMS={np.sqrt(np.mean(signal**2)):.4f}, Peak={np.max(np.abs(signal)):.4f}")

# Test Whisper
print(f"\nTesting Whisper transcription...")
try:
    result = transcriber_debug.model.transcribe(
        signal,
        language="vi",
        fp16=False,
        task="transcribe",
        verbose=False
    )
    detected = result["text"].strip()
    confidence = result.get("confidence", 0)
    print(f"✓ Result: '{detected}'")
    print(f"  (This is expected to be empty since we used sine wave)")
    print(f"\n✅ Whisper model is working!")
    print(f"   → If later RMS > 0.05, the issue is device capture, not Whisper")
except Exception as e:
    print(f"❌ Whisper ERROR: {e}")
    print(f"   → Check if Whisper model loaded correctly")

print("="*60 + "\n")


🧪 TEST: Whisper Recognition with Sample Audio

Generated test audio: 3s at 16000Hz
Audio shape: (48000,)
Audio level: RMS=0.4889, Peak=0.9000

Testing Whisper transcription...



🧪 TEST: Whisper Recognition with Sample Audio

Generated test audio: 3s at 16000Hz
Audio shape: (48000,)
Audio level: RMS=0.4889, Peak=0.9000

Testing Whisper transcription...


  0%|          | 0/300 [00:22<?, ?frames/s]


🧪 TEST: Whisper Recognition with Sample Audio

Generated test audio: 3s at 16000Hz
Audio shape: (48000,)
Audio level: RMS=0.4889, Peak=0.9000

Testing Whisper transcription...


  0%|          | 0/300 [00:22<?, ?frames/s]

✓ Result: ''
  (This is expected to be empty since we used sine wave)

✅ Whisper model is working!
   → If later RMS > 0.05, the issue is device capture, not Whisper



In [31]:
# ===== CREATE & TEST WITH REAL SPEECH FILE =====
# Tải sample Vietnamese audio và test Whisper transcription

import urllib.request
import os
from pathlib import Path

print("\n" + "="*60)
print("📝 TEST: Real Vietnamese Speech File")
print("="*60)

# Tạo thư mục test nếu chưa có
test_audio_dir = Path("test_audio")
test_audio_dir.mkdir(exist_ok=True)

test_file = test_audio_dir / "test_vietnamese.wav"

print(f"\nTest audio file: {test_file}")

# Option 1: Tạo audio từ TTS (nếu có pyttsx3)
try:
    import pyttsx3
    print("🎤 Generating Vietnamese TTS audio...")
    engine = pyttsx3.init()
    engine.setProperty('language', 'vi')
    engine.setProperty('rate', 150)
    engine.save_to_file('Xin chào, đây là bài kiểm tra âm thanh', str(test_file))
    engine.runAndWait()
    print(f"✓ TTS audio created: {test_file}")
except Exception as e:
    print(f"⚠️  TTS not available: {e}")
    print("   Skipping file-based test")
    test_file = None

# Option 2: If test_file created, transcribe it
if test_file and test_file.exists():
    print(f"\nTranscribing {test_file}...")
    try:
        result = transcriber_debug.model.transcribe(
            str(test_file),
            language="vi",
            fp16=False,
            verbose=False
        )
        detected = result["text"].strip()
        print(f"✓ Transcription: '{detected}'")
        print(f"\n✅ File-based transcription works!")
        print(f"   → If this worked, device capture is the issue")
        print(f"   → Solution: Enable Stereo Mix & select correct device")
    except Exception as e:
        print(f"❌ Transcription failed: {e}")
else:
    print("\n⚠️  Skipping file transcription test")
    print("   (pyttsx3 not available - not critical)")

print("="*60 + "\n")


📝 TEST: Real Vietnamese Speech File

Test audio file: test_audio\test_vietnamese.wav
⚠️  TTS not available: No module named 'pyttsx3'
   Skipping file-based test

⚠️  Skipping file transcription test
   (pyttsx3 not available - not critical)



In [ ]:
# ===== INTERACTIVE DEVICE SELECTOR WITH TESTING =====

print("\n" + "="*60)
print("INTERACTIVE DEVICE SELECTOR")
print("="*60)

print("\nAvailable Recording Devices:")
print("-" * 65)
devices = sd.query_devices()
recording_devices = []
for i, device in enumerate(devices):
    if device['max_input_channels'] > 0:
        recording_devices.append(i)
        native = int(device['default_samplerate'])
        marker = "<- DEFAULT" if i == sd.default.device[0] else ""
        print(f"[{i:2d}] {device['name']:42s} Ch:{device['max_input_channels']} SR:{native} {marker}")
print("-" * 65)

# <- SET DEVICE ID HERE TO TEST
DEVICE_TO_TEST = 7

device_info = sd.query_devices(DEVICE_TO_TEST)
num_channels = get_device_channels(DEVICE_TO_TEST)
native_sr = get_native_sr(DEVICE_TO_TEST)

print(f"\nTesting Device [{DEVICE_TO_TEST}]: {device_info['name']}")
print(f"   Channels: {num_channels} | Native SR: {native_sr} Hz")
print(f"\nRecording 5 seconds... (play audio now!)")

try:
    audio_data = sd.rec(
        int(5 * native_sr),
        samplerate=native_sr,
        channels=num_channels,
        device=DEVICE_TO_TEST,
        dtype='float32'
    )
    sd.wait()

    audio_mono = audio_to_mono(audio_data)
    if native_sr != 16000:
        audio_mono = resample_audio(audio_mono, native_sr, 16000)

    rms = np.sqrt(np.mean(audio_mono**2))
    peak = np.max(np.abs(audio_mono))

    status_label = "NO AUDIO" if rms < 0.01 else ("QUIET" if rms < 0.05 else "HAS AUDIO")
    print(f"\nRecording done! ({num_channels}ch @ {native_sr}Hz -> resampled to 16kHz)")
    print(f"  RMS:  {rms:.6f}  [{status_label}]")
    print(f"  Peak: {peak:.6f}")

    if rms < 0.01:
        print(f"\n  Device [{DEVICE_TO_TEST}] khong capture duoc audio")
        print(f"  -> Thu device ID khac")
        print(f"  -> Dam bao YouTube/Spotify dang phat am thanh")
    elif rms < 0.05:
        print(f"\n  Audio rat nho -> tang volume YouTube/app len")
    else:
        print(f"\n  Device [{DEVICE_TO_TEST}] capture audio tot!")
        print(f"  -> Cap nhat MANUAL_DEVICE_ID = {DEVICE_TO_TEST} o cell tren")

    if rms > 0.02:
        print(f"\nWhisper transcription on last 3s...")
        last_3s = audio_mono[-int(3*16000):]
        result = transcriber_debug.model.transcribe(
            last_3s, language="vi", fp16=False, verbose=False
        )
        detected = result["text"].strip()
        print(f"  -> {'Detected: ' + detected if detected else 'No speech (music or silence)'}")

except Exception as e:
    print(f"Error: {e}")
    if "-9998" in str(e):
        print(f"  -> Channel mismatch! Detected channels: {device_info['max_input_channels']}")
    elif "sample rate" in str(e).lower() or "rate" in str(e).lower():
        print(f"  -> Sample rate issue! Native SR: {native_sr} Hz")

print("="*60 + "\n")


🎛️  INTERACTIVE DEVICE SELECTOR

📟 Available Recording Devices:
------------------------------------------------------------
[ 5] CABLE Output (VB-Audio Point)            (Ch: 16) 
[ 7] Input (VB-Audio Point)                   (Ch: 16) 
------------------------------------------------------------

📋 HƯỚNG DẪN SỬ DỤNG:
1. Mở YouTube/Spotify/ứng dụng phát âm thanh khác
2. Bật volume và để phát audio
3. Thay thế DEVICE_TO_TEST = 0 với device ID từ danh sách trên
4. Chạy cell này để test device đó

💡 TİPS:
   - Stereo Mix thường là device cuối cùng
   - Nếu không thấy 'Stereo Mix': Enable nó trong Sound Settings
   - Thử device #0, #1, #2, ... cho đến khi thấy RMS > 0.05


🔧 Testing Device [1]...
   Device: Remote Audio

⏱️  Recording 5 seconds while audio is playing...
❌ Error: Error opening InputStream: Invalid number of channels [PaErrorCode -9998]
   → Device might be in use or invalid

